# AAV2 (Modelization_V1 `aav_viability_test/aav2.csv`) — joint Potts regression ground truth

Same method as `AAV9_potts_regression.ipynb` (`AAVs dataset/AAV9/viability/`) and Part A of
`AAV9_fit4function_potts_vs_mlp.ipynb`: a single joint, ridge-regularized linear (Potts)
regression fit directly on an observed `(sequence, target)` table via
`RegressionV1.fit_weights_potts_from_data` — no round-simulation, no sequential
group-means-then-residual-J, F and J solved together.

**Not the same `aav2.csv` as `Selectivity/AAV2/`.** This notebook uses
`Modelization_V1/notebooks/aav_viability_test/aav2.csv` (53,382 sequences, columns
`sequence,target,error,plasmid,vector,plasmid_norm,vector_norm` — the same family as `aav9.csv`,
already a single flat viability table with a precomputed `target` log-enrichment column), copied
here for self-containment (same convention as `aav9.csv`/`fit4functionaav9.csv` when
`Modelization_V2` was created — never read cross-repo from `Modelization_V1`, gitignored either
way, published on the `aav-raw-ngs-data-v1` GitHub release). This is **not** the confidential IDV
organoid `AAV2_organoides.csv` used by `Selectivity/AAV2/viability/` — that dataset already has
its own Potts fit (`AAV2_viab_sorting.ipynb`, exported as `aav2_F_viab_potts_sorted_cv.npy`). This
notebook's export is intentionally named without a suffix (`aav2_F_viab_potts.npy`,
`aav2_J_viab_potts.npy`) to mirror `aav9_F_viab_potts.npy` — the canonical Potts GT built directly
from the eponymous `aav2.csv`, the same relationship `AAV9_potts_regression.ipynb` has to
`aav9.csv`.

No double-mutant-scan anywhere in this notebook (consistent with the rest of `Modelization_V2`).

### 0. Setup

In [ ]:
import sys, os
from pathlib import Path

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # must be set before jax initializes (RegressionV1 imports jax)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Location-independent lib/ resolution (search upward for the Modelization_V2 folder) -- the
# older AAV9_potts_regression.ipynb hardcodes "../../lib", which no longer resolves correctly
# after the notebooks/notebooks/Viability -> AAVs dataset/AAV9/viability moves (documented,
# unfixed bug in CLAUDE.md); this notebook uses the same robust pattern already adopted by
# every Selectivity/AAV{2,5} notebook instead, so it survives any future reorg.
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "Modelization_V2")
sys.path.insert(0, str(_root / "lib"))
LIB = _root / "lib"

import RegressionV1 as R
from analysisV1 import AA_LABELS, pearson, precision_at_k, plot_cv_curve, plot_topk_recovery

message = "AAV2 (aav_viability_test) joint Potts regression 1.0"
print(message)

### 1. Load `aav2.csv` (same encoding recipe as `AAV9_potts_regression.ipynb`)

In [ ]:
CSV_PATH = Path("aav2.csv")
if not CSV_PATH.exists():
    CSV_PATH = _root / "notebooks/notebooks/AAVs dataset/AAV2/viability/aav2.csv"
assert CSV_PATH.exists(), CSV_PATH

df = pd.read_csv(CSV_PATH)
print(f"{len(df):,} variants, columns: {list(df.columns)}")

lengths = df["sequence"].str.len().unique()
assert len(lengths) == 1, f"expected a single sequence length, got {lengths}"
num_positions = int(lengths[0])
num_amino_acids = len(AA_LABELS)

alphabet = sorted(set("".join(df["sequence"])))
assert alphabet == AA_LABELS, f"alphabet mismatch: {alphabet} vs {AA_LABELS}"

lut = np.zeros(256, dtype=np.int64)
for i, aa in enumerate(AA_LABELS):
    lut[ord(aa)] = i

seq_bytes = np.frombuffer("".join(df["sequence"]).encode("ascii"), dtype=np.uint8)
seq_matrix = lut[seq_bytes].reshape(len(df), num_positions)
target = df["target"].to_numpy(dtype=np.float64)

print(f"seq_matrix shape: {seq_matrix.shape}  (num_sequences, num_positions)")
print(f"error column: {'constant' if df['error'].nunique() == 1 else 'variable'} "
      f"({df['error'].nunique():,} unique values) -- real per-sequence uncertainty, unlike "
      f"aav9.csv's constant 0.1 placeholder")

### 2. Sample weights — inverse-variance from raw counts (`eps=0.5`), not the CSV's `error` column

`fit_weights_potts_from_data`'s own docstring suggests `sample_weight = 1/error**2` for datasets
with a real per-sequence uncertainty column like this one. Checked first, rejected: `error` is
heavy-tailed enough (`min=0.001`, most of the mass at `1.0`-`1.67`) that `1/error**2` is dominated
by a tiny handful of rows -- the top 100 of 53,382 rows (0.19%) alone carry **45%** of the total
weight, top 1,000 (1.9%) carry **84%** -- effectively fitting on a few hundred sequences and
ignoring the rest. Exactly the failure mode already hit and abandoned once in this project
(`AAV5_SEL_profile_model_sel_org2_invvar.ipynb`, unbounded inverse-variance weight dominated by a
few extreme rows, cf. `CLAUDE.md`).

Used instead: the project's standard inverse-variance-from-counts formula (`eps=0.5`, cf.
`CLAUDE.md` permanent convention), `w = 1/(1/(vector+0.5) + 1/(plasmid+0.5))`, computed from the
CSV's own raw `plasmid`/`vector` counts. Far better behaved (top 100 rows carry 11% of the mass,
top 1,000 carry 43%), and independently justified here: `target` reconstructs almost exactly as
`log2((vector+0.5)/(plasmid+0.5))` from these same raw counts (Pearson r checked below) --
confirming `target` already *is* this project's standard `eps=0.5` log2 enrichment, so weighting
by the same counts is the internally consistent choice, not an ad hoc substitute for `error`.

In [ ]:
plasmid = df["plasmid"].to_numpy(dtype=np.float64)
vector  = df["vector"].to_numpy(dtype=np.float64)
error   = df["error"].to_numpy(dtype=np.float64)

w_error  = 1.0 / error**2
w_counts = 1.0 / (1.0 / (vector + 0.5) + 1.0 / (plasmid + 0.5))

def top_mass_share(w, n):
    return float(np.sort(w)[-n:].sum() / w.sum())

print("weight source        top-100 share   top-1000 share")
print(f"  1/error**2            {top_mass_share(w_error, 100):.1%}           {top_mass_share(w_error, 1000):.1%}")
print(f"  count eps=0.5 invvar  {top_mass_share(w_counts, 100):.1%}           {top_mass_share(w_counts, 1000):.1%}")

recon = np.log2((vector + 0.5) / (plasmid + 0.5))
r_recon = pearson(target, recon)
print(f"\nr(target, log2((vector+0.5)/(plasmid+0.5))) = {r_recon:+.4f}  "
      f"(confirms target IS this project's eps=0.5 log2 enrichment)")

sample_weight = w_counts
print(f"\nUsing count-based eps=0.5 inverse-variance as sample_weight "
      f"(mean={sample_weight.mean():.2f}, median={np.median(sample_weight):.2f}, "
      f"max={sample_weight.max():.2f})")

### 3. Joint Potts ridge regression on the full dataset (weighted)

`fit_weights_potts_from_data` builds the same single-site + pairwise one-hot design as
`build_potts_features` (140 + 8,400 = 8,540 features + bias), picks the ridge strength by 5-fold
CV over `RegressionV1`'s default `lambdas_grid = np.logspace(-1, 2, 30)` (weighted CV loop and
final fit, since `sample_weight` is passed), then refits on the full data at that lambda.

In [ ]:
F_potts, J_potts, rank_full, info_full = R.fit_weights_potts_from_data(
    seq_matrix, target, sample_weight=sample_weight, seed=0,
)

In [ ]:
fig = plot_cv_curve(info_full["lambdas_grid"], info_full["cv_mse"],
                     title=f"Ridge CV -- full aav2.csv ({len(df):,} sequences, weighted)")
plt.show()

### 4. Held-out predictive check: naive group-means vs Potts regression

No pre-existing naive GT file for this exact dataset to load (unlike `aav9.csv`'s
`load_F_viab_aav9_mlp`/`load_J_viab_aav9_mlp`), so both methods are fit locally on the same
`idx_train` split and scored on `idx_test` -- same 50/50 split convention as
`AAV9_potts_regression.ipynb`/`AAV9_profile_model.ipynb`
(`test_size=0.5, random_state=0`). Naive group-means helpers reproduced verbatim from
`AAV9_potts_regression.ipynb` (itself reproduced from `AAV9_profile_model.ipynb` cells 11/24 --
notebook-local, not shared via `lib/`, same convention every `AAV{2,5,9}_profile_model.ipynb`
already follows). **Caveat**: the naive baseline below is unweighted (plain per-cell mean, the
simplest form of "naive extraction") while the Potts regression is weighted (section 2) -- the
comparison partly reflects that choice, not purely joint-vs-sequential fitting; kept unweighted to
match the original naive-extraction convention rather than inventing a weighted variant.

In [ ]:
def F_groundtruth_viability(seq_matrix, target, num_amino_acids, num_positions):
    baseline = target.mean()
    F = np.full((num_amino_acids, num_positions), np.nan)
    counts = np.zeros((num_amino_acids, num_positions), dtype=int)
    for i in range(num_positions):
        for aa in range(num_amino_acids):
            mask = seq_matrix[:, i] == aa
            counts[aa, i] = mask.sum()
            if mask.any():
                F[aa, i] = target[mask].mean() - baseline
    return F, counts


def J_groundtruth_naive(seq_matrix, target, F, baseline, num_amino_acids, num_positions, min_support=5):
    L, A = num_positions, num_amino_acids
    J       = np.full((L, L, A, A), np.nan)
    J_raw   = np.full((L, L, A, A), np.nan)
    J_se    = np.full((L, L, A, A), np.inf)
    support = np.zeros((L, L, A, A), dtype=int)

    position_masks = [[seq_matrix[:, i] == a for a in range(A)] for i in range(L)]

    for i in range(L):
        for j in range(i + 1, L):
            for a in range(A):
                mask_a = position_masks[i][a]
                if not mask_a.any():
                    continue
                for b in range(A):
                    mask = mask_a & position_masks[j][b]
                    n = int(mask.sum())
                    support[i, j, a, b] = support[j, i, b, a] = n
                    if n >= 1:
                        vals_t = target[mask]
                        val = vals_t.mean() - baseline - F[a, i] - F[b, j]
                        J_raw[i, j, a, b] = J_raw[j, i, b, a] = val
                        if n >= 2:
                            se = vals_t.std(ddof=1) / np.sqrt(n)
                            J_se[i, j, a, b] = J_se[j, i, b, a] = se
                        if n >= min_support:
                            J[i, j, a, b] = J[j, i, b, a] = val
    return J, support, J_raw, J_se


def score_FJ(seq, F, J, baseline):
    L = seq.shape[1]
    F_part = F[seq, np.arange(L)].sum(axis=1).astype(np.float64)
    J_part = np.zeros(len(seq), dtype=np.float64)
    for i in range(L):
        for j in range(i + 1, L):
            J_part += J[i, j, seq[:, i], seq[:, j]]
    return F_part, J_part, baseline + F_part + J_part

In [ ]:
idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.5, random_state=0)
baseline_tr = float(target[idx_train].mean())

F_potts_tr, J_potts_tr, rank_tr, info_tr = R.fit_weights_potts_from_data(
    seq_matrix[idx_train], target[idx_train], sample_weight=sample_weight[idx_train],
    seed=0, verbose=False,
)

F_naive_tr, _counts_tr = F_groundtruth_viability(
    seq_matrix[idx_train], target[idx_train], num_amino_acids, num_positions,
)
J_naive_tr, _support_tr, _J_raw_tr, _J_se_tr = J_groundtruth_naive(
    seq_matrix[idx_train], target[idx_train], F_naive_tr, baseline=baseline_tr,
    num_amino_acids=num_amino_acids, num_positions=num_positions, min_support=5,
)
J_naive_tr = np.nan_to_num(J_naive_tr, nan=0.0)
F_naive_tr = np.nan_to_num(F_naive_tr, nan=0.0)

_, _, pred_potts_test = score_FJ(seq_matrix[idx_test], np.array(F_potts_tr), np.array(J_potts_tr), baseline_tr)
_, _, pred_naive_test = score_FJ(seq_matrix[idx_test], F_naive_tr, J_naive_tr, baseline_tr)

r_naive = pearson(target[idx_test], pred_naive_test)
r_potts = pearson(target[idx_test], pred_potts_test)

print(f"Held-out (idx_test, {len(idx_test):,} sequences) Pearson r -- real target vs prediction, "
      f"both fit on idx_train ({len(idx_train):,} sequences) only:")
print(f"  naive (group-means, min_support=5 hard cutoff, unweighted) : {r_naive:+.4f}")
print(f"  Potts regression (ridge, CV lambda={info_tr['lam']:.3g}, weighted) : {r_potts:+.4f}")

### 5. Percentile recovery (top-k%) on the held-out set

Same sweep as `AAV9_fit4function_potts_vs_mlp.ipynb` section 4b: for each percentile k, does the
Potts regression's own top-k% (by predicted score, `idx_train`-fit) recover the REAL top-k% (ranked
by the held-out `target` measurement)?

In [ ]:
PERCENTILES = [1, 2, 5, 10, 20, 30, 50, 70, 100]

def recovery_table(y_true_test, y_pred_test, n):
    rows = []
    for pct in PERCENTILES:
        k_frac = pct / 100
        k = max(1, int(n * k_frac))
        rows.append(dict(percentile=pct, k=k,
                          recovery=precision_at_k(y_true_test, y_pred_test, k_frac=k_frac),
                          random_baseline=k_frac))
    return pd.DataFrame(rows).set_index("percentile")

potts_recovery_df = recovery_table(target[idx_test], pred_potts_test, len(idx_test))
naive_recovery_df = recovery_table(target[idx_test], pred_naive_test, len(idx_test))
print("Potts regression: recovery of the real top-k% within the model's own top-k%:")
display(potts_recovery_df)

fig = plot_topk_recovery(target[idx_test], pred_potts_test, k_frac=0.10,
                          xlabel="real target (held-out)", ylabel="Potts prediction",
                          title=f"Potts regression -- top-10% recovery (n={len(idx_test):,})")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.plot(potts_recovery_df.index, potts_recovery_df["recovery"], marker="o", color="tab:green", label="Potts regression")
ax.plot(naive_recovery_df.index, naive_recovery_df["recovery"], marker="s", color="tab:gray", label="naive (group-means)")
ax.plot(potts_recovery_df.index, potts_recovery_df["random_baseline"], linestyle="--", color="gray",
        label="random-guessing baseline")
ax.set_xlabel("percentile (top-k%, ranked by real target)")
ax.set_ylabel("recovery (fraction of real top-k% also in the model's own top-k%)")
ax.set_ylim(0, 1.02)
ax.set_title(f"Percentile recovery, held-out set (n={len(idx_test):,})")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

### 6. Credibility check: does the brute-force top-500 show the F/J rescue pathology?

Same lightweight diagnostic as `AAV9_potts_regression.ipynb` section 5 / `AAV9_profile_model.ipynb`
section 3c: score 2,000,000 random sequences drawn uniformly from the full
`20**7 ~= 1.28e9`-sequence space with the FULL-data `F_potts`/`J_potts` (section 3, not the
held-out split), and check whether the top-500 lean on a large positive `J_part` to rescue a bad
`F_part` -- the pattern flagged elsewhere in this project as not credible.

In [ ]:
rng_bf = np.random.default_rng(0)
N_BF, TOP_K_BF = 2_000_000, 500
seq_bf = rng_bf.integers(0, num_amino_acids, size=(N_BF, num_positions))
baseline_full = float(target.mean())

F_part, J_part, total = score_FJ(seq_bf, np.array(F_potts), np.array(J_potts), baseline_full)
top_idx = np.argsort(-total)[:TOP_K_BF]
print("Potts regression (full aav2.csv, weighted):")
print(f"  top-{TOP_K_BF}      F_part mean={F_part[top_idx].mean():+.3f}  "
      f"J_part mean={J_part[top_idx].mean():+.3f}  "
      f"({(J_part[top_idx] > 0).mean() * 100:.1f}% positive J_part)")
print(f"  random background  F_part mean={F_part.mean():+.3f}  "
      f"J_part mean={J_part.mean():+.3f}  "
      f"({(J_part > 0).mean() * 100:.1f}% positive J_part)")

### 7. Export `aav2_F_viab_potts.npy` / `aav2_J_viab_potts.npy`

Full-data fit (section 3), not the `idx_train`-only fit from section 4 (that one is for held-out
validation only). Named without a suffix -- mirrors `aav9_F_viab_potts.npy`/`aav9_J_viab_potts.npy`,
the canonical Potts GT built directly from the eponymous CSV -- distinct from
`aav2_F_viab_potts_sorted_cv.npy`/`aav2_J_viab_potts_sorted_cv.npy` (fit on the unrelated IDV
organoid `AAV2_organoides_sorted.csv` in `Selectivity/AAV2/viability/sorting/`). Both are
gitignored (`Modelization_V2/lib/aav2_*_potts*.npy`), no `initialize_weights.py` loader added yet
(none of the `aav2_*_potts*` variants have one so far -- consistent with the rest of the project).

In [ ]:
F_VIAB_POTTS_PATH = LIB / "aav2_F_viab_potts.npy"
J_VIAB_POTTS_PATH = LIB / "aav2_J_viab_potts.npy"

np.save(F_VIAB_POTTS_PATH, np.array(F_potts, dtype=np.float32))
np.save(J_VIAB_POTTS_PATH, np.array(J_potts, dtype=np.float32))

print(f"Saved F_potts {np.array(F_potts).shape} to {F_VIAB_POTTS_PATH}")
print(f"Saved J_potts {np.array(J_potts).shape} to {J_VIAB_POTTS_PATH}")
print(f"(fit on all {len(df):,} sequences, CV lambda={info_full['lam']:.3g}, "
      f"held-out r={r_potts:+.4f} from section 4)")